# Lesson 4 exercise: a dataset for chat

**Goal:** turn the extraction task into a training set and a held-out test set that TRL (Hugging Face's library for fine-tuning chat models) can read, and see the exact text supervised fine-tuning (SFT) will train on. No training today.

**How to use this notebook**
1. In Colab, any runtime works today. A CPU runtime is fine, because you only load the tokenizer.
2. Run the cells top to bottom.
3. Wherever you see `...` after a `# TODO`, replace it with your code.
4. Run the **check** cell under each TODO. ✅ means you've got it; ❌ gives you a hint.

Stuck? Look up the error in the course's troubleshooting sheet (`reference/troubleshooting.html`). Still stuck? Ask on the Hugging Face Forums with your code and the full traceback.

## 0 · Setup
Installs the course's library versions (about 30 seconds) and defines a tiny `check()` helper.

In [ ]:
%pip install -q -U "transformers>=5.19,<6" "trl>=1.15,<2" "peft>=0.21,<1" "datasets>=5,<6" "accelerate>=1.15,<2"
import torch, transformers
device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device:", device)

def check(passed, ok_msg, hint):
    print(("✅ " + ok_msg) if passed else ("❌ " + hint))

## 1 · The task data
This cell is shared by every lesson from here on. `make_rows(n, split, seed)` returns `n` rows in TRL's **prompt-completion** format: the user's message in `"prompt"`, the ideal assistant reply in `"completion"`.
`split="test"` draws from a separate pool of names, cities and phrasings that never appear in training.

In [ ]:
import json, random

PROMPT = "Extract the person's name, age and city as JSON."
NAMES = ["Sara", "Omar", "Lina", "Youssef", "Mona", "Karim", "Nour", "Emma", "Liam", "Sofia",
         "Mateo", "Aiko", "Kenji", "Priya", "Arjun", "Chloe", "Lucas", "Fatima", "Diego", "Elena",
         "Leo", "Amara", "Kofi", "Hana", "Ines"]          # last 5 are test-only
CITIES = ["Cairo", "Alexandria", "Paris", "Lagos", "Tokyo", "Lima", "Berlin", "Toronto",
          "Mumbai", "Seoul", "Nairobi", "Madrid", "Dubai", "Oslo", "Austin", "Lisbon",
          "Hanoi", "Accra", "Quito", "Perth"]                # last 4 are test-only
TEMPLATES = [
    "Hi, I'm {name} from {city}. I'm {age}.",
    "{name} ({age}) lives in {city}.",
    "My name is {name}, I am {age} years old and I live in {city}.",
    "New signup: {name}, age {age}, based in {city}.",
    "Hello from {city}! {name} here, just turned {age}.",
    "{city}-based {name}, {age}, is joining the team next week.",
    "Contact: {name} | {age} | {city}",
    "Applicant {name} is {age} and currently resides in {city}.",
    "Meet {name}: {age} years young and loving life in {city}.",   # test-only
    "I moved to {city} last year. Name's {name}, age {age}.",      # test-only
]

def make_rows(n, split="train", seed=0):
    """Return n chat examples in TRL's prompt-completion format."""
    rng = random.Random(seed)
    test = split == "test"
    names = NAMES[20:] if test else NAMES[:20]
    cities = CITIES[16:] if test else CITIES[:16]
    templates = TEMPLATES[8:] if test else TEMPLATES[:8]
    rows = []
    for _ in range(n):
        person = {"name": rng.choice(names), "age": rng.randint(18, 80), "city": rng.choice(cities)}
        text = rng.choice(templates).format(**person)
        rows.append({
            "prompt": [{"role": "user", "content": f"{PROMPT}\n\nText: {text}"}],
            "completion": [{"role": "assistant", "content": json.dumps(person)}],
        })
    return rows

In [ ]:
rows = make_rows(2, "train", seed=0)
print(type(rows), len(rows))
print(rows[0])

## 2 · From a list of dicts to a `Dataset`
`Dataset.from_list(rows)` turns a list of dicts into a 🤗 Datasets table, with one column per key.

**TODO 1:** Build `train` from **500** training rows with `seed=0`, and `test` from **40** test rows with `seed=1`. Lesson 5 uses exactly these sizes and seeds.

In [ ]:
from datasets import Dataset

# TODO 1
train = ...
test = ...

print(train)
print(test)

In [ ]:
# check 1
ok = (isinstance(train, Dataset) and isinstance(test, Dataset)
      and len(train) == 500 and len(test) == 40
      and train.column_names == ["prompt", "completion"]
      and all(json.loads(c[0]["content"])["name"] in NAMES[20:] for c in test["completion"])
      and train[0] == make_rows(1, "train", seed=0)[0] and test[0] == make_rows(1, "test", seed=1)[0])
check(ok, "500 training rows and 40 held-out test rows.",
      "Wrap make_rows(...) in Dataset.from_list(...). Check the sizes, the split names and the seeds.")

**Indexing works like a table.** `train[0]` is one row (a dict), `train["completion"]` is a whole column, and `train[:2]` is the first two rows as a dict of lists.

In [ ]:
print(train[0])
print(train[:2]["completion"])

**Is the test set really held out?** Collect the names and cities on each side and intersect them.

In [ ]:
def people(ds):
    return [json.loads(c[0]["content"]) for c in ds["completion"]]

train_names = {p["name"] for p in people(train)}
test_names = {p["name"] for p in people(test)}
train_cities = {p["city"] for p in people(train)}
test_cities = {p["city"] for p in people(test)}
print("test names: ", sorted(test_names))
print("names in both splits: ", train_names & test_names)
print("cities in both splits:", train_cities & test_cities)

## 3 · The exact text SFT trains on
SFT never sees your dicts. It joins the prompt and the completion into one conversation and renders it with the **chat template**, the same template you met in Lesson 1.

**TODO 2:** Render `row` as training text. Join the two message lists into one conversation, and ask for **text**, not token ids. Leave `add_generation_prompt` off: the assistant's reply is already part of the conversation.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

row = train[0]

# TODO 2
text = ...

print(text)

In [ ]:
# check 2
reply = row["completion"][0]["content"]
ok = (isinstance(text, str) and "<|im_start|>assistant" in text and reply in text
      and text.rstrip().endswith("<|im_end|>"))
check(ok, "That's the full training text: prompt and completion in one templated string.",
      "Join the two message lists into one conversation, then look at apply_chat_template's `tokenize` argument.")

**Which part does the loss count?** Render the prompt alone, as at inference time. Everything after it is the completion. Because our rows are prompt-completion, TRL masks the prompt tokens with `-100` (Lesson 3) and computes the loss on the completion only.

In [ ]:
prompt_text = tok.apply_chat_template(row["prompt"], tokenize=False, add_generation_prompt=True)
print("is the prompt a prefix of the training text?", text.startswith(prompt_text))
print("completion part (the loss counts only this):", repr(text[len(prompt_text):]))

## 4 · Count tokens with `map`
`map(fn)` runs `fn` on every row and adds the keys of the dict it returns as new columns. It returns a **new** dataset; it doesn't change the old one in place.

`add_length` below counts one row's tokens after templating.

**TODO 3:** Apply `add_length` to every row of `train` and of `test`, and keep the results.

*If you run this cell before filling it in, `train` and `test` become `...`. Rerun the TODO 1 cell to rebuild them.*

In [ ]:
def add_length(row):
    ids = tok.apply_chat_template(row["prompt"] + row["completion"], return_dict=True)["input_ids"]
    return {"n_tokens": len(ids)}

# TODO 3
train = ...
test = ...

print(train)

In [ ]:
# check 3
ok = (isinstance(train, Dataset) and isinstance(test, Dataset)
      and "n_tokens" in train.column_names and "n_tokens" in test.column_names
      and len(train) == 500 and len(test) == 40)
check(ok, "Both splits now carry an n_tokens column.",
      "map returns a new dataset. Did you keep the result, for both splits?")

In [ ]:
longest = max(max(train["n_tokens"]), max(test["n_tokens"]))
print("train tokens: min", min(train["n_tokens"]), "| max", max(train["n_tokens"]))
print("test tokens:  min", min(test["n_tokens"]), "| max", max(test["n_tokens"]))
print("longest row:", longest)

## 5 · Choose `max_length`
In Lesson 5, `SFTConfig(max_length=...)` cuts any row longer than that many tokens, **keeping the start**. The completion sits at the end, so a cap that's too small cuts off the JSON and the `<|im_end|>` that teaches the model to stop.

**TODO 4:** Pick a `max_length` that fits every row with room to spare. A power of two is the usual choice.

In [ ]:
# TODO 4
max_length = ...

In [ ]:
# check 4
ok = isinstance(max_length, int) and longest <= max_length <= 1024
check(ok, f"No row gets truncated: the longest is {longest} tokens.",
      "max_length must be at least the longest row printed above (and no more than 1024 for this task).")

## Done
You have `train` (500 rows) and `test` (40 held-out rows) in the format `SFTTrainer` reads, and you know every row fits your `max_length`. Lesson 5 rebuilds them with the same `make_rows` calls and seeds, then trains on them.

## Stretch (optional, about 5 min)
1. Build the other shape TRL accepts: `{"messages": row["prompt"] + row["completion"]}`. Render its `"messages"` list. Is the text identical to `text`? So what's actually different when you train on it?
2. Add your own phrasing to `TEMPLATES` (rerun the data cell), rebuild `train`, and watch how `n_tokens` changes.
3. Use `train.filter(lambda r: r["n_tokens"] > 88)` to find the longest rows. Which templates are they?